In [2]:
import pandas as pd

df = pd.read_csv("shipment.csv")

print(df.shape)
print(df.head())
print(df.info())
print(df.isna().sum())
print(df["delivery_status"].value_counts())

(728, 12)
     shipment_id    type        date      product_category    origin  \
0  SHP-2024-0001  Export  02/01/2024           Electronics    Mumbai   
1  SHP-2024-0002  Import  03/01/2024              Textiles  Shanghai   
2  SHP-2024-0003  Export  04/01/2024        Consumer Goods    Mumbai   
3  SHP-2024-0004  Import  05/01/2024  Industrial Equipment   Hamburg   
4  SHP-2024-0005  Export  06/01/2024           Electronics    Mumbai   

  O_Country destination D_Country   value  freight_cost  \
0     India    New York       USA   85000          4250   
1     China      Mumbai     India  120000          6000   
2     India      London        UK   45000          2250   
3   Germany      Mumbai     India  250000         12500   
4     India       Tokyo     Japan   95000          4750   

   customs_clearance_time_days delivery_status  
0                          2.1         On-Time  
1                          3.5         On-Time  
2                          1.8         On-Time  
3     

In [4]:
df["date"] = pd.to_datetime(
    df["date"],
    dayfirst=True,
    errors="coerce"
)

if df["date"].isna().any():
    raise ValueError("Some dates could not be parsed.")

print(df["date"].min())
print(df["date"].max())

2024-01-02 00:00:00
2025-12-31 00:00:00


In [5]:
df["month"] = df["date"].dt.month
df["dayofweek"] = df["date"].dt.dayofweek
df["dayofmonth"] = df["date"].dt.day

print(
    df[
        ["date", "month", "dayofweek", "dayofmonth"]
    ].head()
)

        date  month  dayofweek  dayofmonth
0 2024-01-02      1          1           2
1 2024-01-03      1          2           3
2 2024-01-04      1          3           4
3 2024-01-05      1          4           5
4 2024-01-06      1          5           6


In [6]:
FEATURES = [
    "type",
    "product_category",
    "origin",
    "O_Country",
    "destination",
    "D_Country",
    "value",
    "freight_cost",
    "month",
    "dayofweek",
    "dayofmonth"
]

CATEGORICAL = [
    "type",
    "product_category",
    "origin",
    "O_Country",
    "destination",
    "D_Country"
]

NUMERICAL = [
    "value",
    "freight_cost",
    "month",
    "dayofweek",
    "dayofmonth"
]

In [7]:
train_mask = df["date"] < pd.Timestamp("2025-01-01")
test_mask = df["date"] >= pd.Timestamp("2025-01-01")

X = df[FEATURES]
y = df["customs_clearance_time_days"]

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]

y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

print(X_train.shape)
print(X_test.shape)

(364, 11)
(364, 11)


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer([
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        CATEGORICAL
    ),
    (
        "numerical",
        StandardScaler(),
        NUMERICAL
    )
])

In [9]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

regression_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        RandomForestRegressor(
            n_estimators=500,
            min_samples_leaf=3,
            random_state=42,
            n_jobs=-1
        )
    )
])

regression_pipeline.fit(X_train, y_train)

predicted_days = regression_pipeline.predict(X_test)

In [10]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, predicted_days)
rmse = mean_squared_error(
    y_test,
    predicted_days
) ** 0.5
r2 = r2_score(y_test, predicted_days)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 0.08087302963780106
RMSE: 0.1046540736390387
R²: 0.9882209196194379


In [11]:
RISK_THRESHOLD_DAYS = 4.0

y_risk = (
    df["customs_clearance_time_days"]
    > RISK_THRESHOLD_DAYS
).astype(int)

print(y_risk.value_counts(normalize=True))

customs_clearance_time_days
0    0.615385
1    0.384615
Name: proportion, dtype: float64


In [12]:
from sklearn.ensemble import RandomForestClassifier

classification_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=4,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    )
])

classification_pipeline.fit(X_train, y_risk.loc[train_mask])

risk_probability = (
    classification_pipeline
    .predict_proba(X_test)[:, 1]
)

In [13]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

risk_prediction = (risk_probability >= 0.50).astype(int)

print("ROC-AUC:", roc_auc_score(y_risk.loc[test_mask], risk_probability))
print(
    "Average Precision:",
    average_precision_score(
        y_risk.loc[test_mask],
        risk_probability
    )
)
print(
    "Precision:",
    precision_score(
        y_risk.loc[test_mask],
        risk_prediction
    )
)
print(
    "Recall:",
    recall_score(
        y_risk.loc[test_mask],
        risk_prediction
    )
)
print(
    "F1:",
    f1_score(
        y_risk.loc[test_mask],
        risk_prediction
    )
)
print(
    "Confusion matrix:",
    confusion_matrix(
        y_risk.loc[test_mask],
        risk_prediction
    )
)

ROC-AUC: 0.9959183673469388
Average Precision: 0.9930440707230962
Precision: 0.9436619718309859
Recall: 0.9571428571428572
F1: 0.950354609929078
Confusion matrix: [[216   8]
 [  6 134]]


In [16]:
from pathlib import Path
import joblib

# Create the model folder if it doesn't already exist
MODEL_DIR = Path("model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save the models
bundle = {
    "regression_model": regression_pipeline,
    "classification_model": classification_pipeline,
    "feature_columns": FEATURES,
    "risk_threshold_days": RISK_THRESHOLD_DAYS,
    "model_version": "1.0"
}

model_path = MODEL_DIR / "logistics_clearance_models.joblib"

joblib.dump(bundle, model_path)

print(f"Model saved successfully to: {model_path.resolve()}")

Model saved successfully to: C:\Users\BLESSETH\Downloads\Logistics\Supply Chain\model\logistics_clearance_models.joblib


In [17]:
from pathlib import Path

# Create the model folder in the same location as your notebook
MODEL_DIR = Path.cwd() / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Current working directory:")
print(Path.cwd())

print("\nModel folder:")
print(MODEL_DIR)

print("\nFolder created successfully!")

Current working directory:
C:\Users\BLESSETH\Downloads\Logistics\Supply Chain

Model folder:
C:\Users\BLESSETH\Downloads\Logistics\Supply Chain\model

Folder created successfully!


In [18]:
import joblib

bundle = {
    "regression_model": regression_pipeline,
    "classification_model": classification_pipeline,
    "feature_columns": FEATURES,
    "risk_threshold_days": RISK_THRESHOLD_DAYS,
    "model_version": "1.0"
}

model_path = MODEL_DIR / "logistics_clearance_models.joblib"

joblib.dump(bundle, model_path)

print("✅ Model saved successfully!")
print(f"📁 Location: {model_path}")

✅ Model saved successfully!
📁 Location: C:\Users\BLESSETH\Downloads\Logistics\Supply Chain\model\logistics_clearance_models.joblib


In [19]:
new_shipment = pd.DataFrame([{
    "type": "Export",
    "product_category": "Electronics",
    "origin": "Mumbai",
    "O_Country": "India",
    "destination": "New York",
    "D_Country": "USA",
    "value": 85000,
    "freight_cost": 4250,
    "date": pd.Timestamp("2026-08-26")
}])

new_shipment["month"] = new_shipment["date"].dt.month
new_shipment["dayofweek"] = new_shipment["date"].dt.dayofweek
new_shipment["dayofmonth"] = new_shipment["date"].dt.day

model_input = new_shipment[FEATURES]

predicted_days = regression_pipeline.predict(
    model_input
)[0]

risk_probability = (
    classification_pipeline
    .predict_proba(model_input)[0, 1]
)

print(f"Predicted clearance: {predicted_days:.2f} days")
print(f"High-risk probability: {risk_probability:.1%}")

Predicted clearance: 3.29 days
High-risk probability: 23.1%


In [21]:
# Get the shipment value from the new shipment
shipment_value = new_shipment["value"].iloc[0]

# Calculate estimated value exposed to clearance risk
value_at_risk = shipment_value * risk_probability

print(
    f"Estimated value exposed to risk: "
    f"${value_at_risk:,.0f}"
)

Estimated value exposed to risk: $19,616


In [24]:
import joblib
import streamlit as st

@st.cache_resource
def load_models():
    return joblib.load(
        "model/logistics_clearance_models.joblib"
    )

bundle = load_models()

regression_model = bundle["regression_model"]
classification_model = bundle["classification_model"]

In [26]:
import pandas as pd
import numpy as np

# Load the shipment data as the batch we want to score
batch = pd.read_csv("shipment.csv")

# Convert the shipment date
batch["date"] = pd.to_datetime(
    batch["date"],
    dayfirst=True,
    errors="coerce"
)

# Check for invalid dates
if batch["date"].isna().any():
    raise ValueError("Some shipment dates could not be parsed.")

# Create the same features used during model training
batch["month"] = batch["date"].dt.month
batch["dayofweek"] = batch["date"].dt.dayofweek
batch["dayofmonth"] = batch["date"].dt.day

print("Batch data loaded successfully.")
print(f"Rows: {len(batch)}")
print(batch.head())

Batch data loaded successfully.
Rows: 728
     shipment_id    type       date      product_category    origin O_Country  \
0  SHP-2024-0001  Export 2024-01-02           Electronics    Mumbai     India   
1  SHP-2024-0002  Import 2024-01-03              Textiles  Shanghai     China   
2  SHP-2024-0003  Export 2024-01-04        Consumer Goods    Mumbai     India   
3  SHP-2024-0004  Import 2024-01-05  Industrial Equipment   Hamburg   Germany   
4  SHP-2024-0005  Export 2024-01-06           Electronics    Mumbai     India   

  destination D_Country   value  freight_cost  customs_clearance_time_days  \
0    New York       USA   85000          4250                          2.1   
1      Mumbai     India  120000          6000                          3.5   
2      London        UK   45000          2250                          1.8   
3      Mumbai     India  250000         12500                          4.2   
4       Tokyo     Japan   95000          4750                          2.5   

  

In [27]:
# Select exactly the features used during training
model_input = batch[FEATURES]

# Predict customs clearance time
batch["predicted_clearance_days"] = (
    regression_pipeline.predict(model_input)
)

# Predict probability of high clearance risk
batch["high_clearance_probability"] = (
    classification_pipeline.predict_proba(model_input)[:, 1]
)

print("Batch predictions completed successfully.")

Batch predictions completed successfully.


In [28]:
batch["clearance_risk"] = np.select(
    [
        batch["high_clearance_probability"] >= 0.70,
        batch["high_clearance_probability"] >= 0.40
    ],
    [
        "High",
        "Medium"
    ],
    default="Low"
)

print(
    batch[
        [
            "shipment_id",
            "predicted_clearance_days",
            "high_clearance_probability",
            "clearance_risk"
        ]
    ].head(10)
)

     shipment_id  predicted_clearance_days  high_clearance_probability  \
0  SHP-2024-0001                  2.248165                    0.026192   
1  SHP-2024-0002                  3.448849                    0.201615   
2  SHP-2024-0003                  1.982250                    0.023683   
3  SHP-2024-0004                  4.361247                    0.974705   
4  SHP-2024-0005                  2.473286                    0.043695   
5  SHP-2024-0006                  2.107802                    0.021162   
6  SHP-2024-0007                  1.811544                    0.023244   
7  SHP-2024-0008                  3.914173                    0.531308   
8  SHP-2024-0009                  2.351202                    0.021139   
9  SHP-2024-0010                  2.853565                    0.030574   

  clearance_risk  
0            Low  
1            Low  
2            Low  
3           High  
4            Low  
5            Low  
6            Low  
7         Medium  
8            L

In [29]:
batch["value_at_risk_usd"] = (
    batch["value"]
    * batch["high_clearance_probability"]
)

print(
    batch[
        [
            "shipment_id",
            "value",
            "predicted_clearance_days",
            "high_clearance_probability",
            "clearance_risk",
            "value_at_risk_usd"
        ]
    ].head(10)
)

     shipment_id   value  predicted_clearance_days  \
0  SHP-2024-0001   85000                  2.248165   
1  SHP-2024-0002  120000                  3.448849   
2  SHP-2024-0003   45000                  1.982250   
3  SHP-2024-0004  250000                  4.361247   
4  SHP-2024-0005   95000                  2.473286   
5  SHP-2024-0006   60000                  2.107802   
6  SHP-2024-0007   75000                  1.811544   
7  SHP-2024-0008  180000                  3.914173   
8  SHP-2024-0009   65000                  2.351202   
9  SHP-2024-0010   90000                  2.853565   

   high_clearance_probability clearance_risk  value_at_risk_usd  
0                    0.026192            Low        2226.291628  
1                    0.201615            Low       24193.828739  
2                    0.023683            Low        1065.715441  
3                    0.974705           High      243676.220744  
4                    0.043695            Low        4151.069718  
5        

In [30]:
output_path = "shipment_predictions.csv"

batch.to_csv(
    output_path,
    index=False
)

print(f"✅ Predictions saved successfully to: {output_path}")

✅ Predictions saved successfully to: shipment_predictions.csv
